<a href="https://colab.research.google.com/github/GayoungJeong/customer-segmentation/blob/main/03_RFM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
clean = pd.read_csv('/content/drive/MyDrive/marketing-portfolio/data/retail_clean.csv')
print(len(clean))   # 824364

Mounted at /content/drive
824364


In [2]:
clean['InvoiceDate'] = pd.to_datetime(clean['InvoiceDate'])

In [3]:
purchases = clean[~clean['Invoice'].astype(str).str.startswith('C')]
print(len(purchases))  #R,F는 실제로 산 기록만 봐야하므로 환불건 삭제

805620


In [4]:
clean.groupby('Customer ID')['TotalPrice'].sum()

,TotalPrice
Customer ID,
12346.0,-64.68
12347.0,5633.32
12348.0,2019.40
12349.0,4404.54
12350.0,334.40
...,...
18283.0,2736.65
18284.0,436.68
18285.0,427.00


In [5]:
ref_date = clean['InvoiceDate'].max() + pd.Timedelta(days=1)
print(ref_date)

2011-12-10 12:50:00


In [6]:
last_purchase = purchases.groupby('Customer ID')['InvoiceDate'].max()   # 가장 늦은 날짜 = 최댓값
recency = (ref_date - last_purchase).dt.days                            # 며칠 차이인지

In [7]:
frequency = purchases.groupby('Customer ID')['Invoice'].nunique() #중복을 빼고 개수가 몇 개인지

In [8]:
monetary = clean.groupby('Customer ID')['TotalPrice'].sum()

In [9]:
rfm = pd.DataFrame({'Recency': recency, 'Frequency': frequency, 'Monetary': monetary})
print(len(rfm))   # 고객 수
rfm.head()

5942


,Recency,Frequency,Monetary
Customer ID,,,
12346.0,326.0,12.0,-64.68
12347.0,2.0,8.0,5633.32
12348.0,75.0,5.0,2019.40
12349.0,19.0,4.0,4404.54
12350.0,310.0,1.0,334.40


In [10]:
rfm.describe()

,Recency,Frequency,Monetary
count,5881.000000,5881.000000,5942.000000
mean,201.457745,6.287196,2801.799459
std,209.474135,13.012879,13973.922553
min,1.000000,1.000000,-25111.090000
25%,26.000000,1.000000,325.097500
50%,96.000000,3.000000,843.970000
75%,380.000000,7.000000,2182.005000
max,739.000000,398.000000,598215.220000


R,F에는 취소건만 있는 고객이 빠짐. Monetary에는 총 고객수가 들어가 있음. 따라서 5942-5881= 61명은 취소건만 있는 고객이고 Monetary에서 추가로 뺄 항목은 머니터리 항목이 0이하인 고객. (샀다가 전부(또는 그 이상) 취소한 고객은 회사에 남긴 매출이 없어서, 가치 점수를 매기는 대상이 아니에요. 다만 **”전량 반품 고객"**은 따로 볼 만한 그룹이에요. 왜 반품했는지 알아보는 건 다른 마케팅 과제) 그래서 그 수가 41명이고, 총 102명 제외(머니터리가 0이하인 수)

In [12]:
low = rfm[rfm['Monetary'] <= 0]   # Monetary가 0 이하인 고객만
print(len(low))                    # 몇 명인지
low.head()

102


,Recency,Frequency,Monetary
Customer ID,,,
12346.0,326.0,12.0,-64.68
12382.0,NaN,NaN,-18.38
12467.0,423.0,1.0,0.00
12503.0,584.0,1.0,0.00
12706.0,NaN,NaN,-91.89


In [13]:
rfm.loc[16446]

,16446.0
Recency,1.0
Frequency,2.0
Monetary,-6.1


In [14]:
has_purchase = rfm[rfm['Recency'].notnull()]          # 구매 기록 있는 고객만
print(len(has_purchase[has_purchase['Monetary'] <= 0]))  # 그중 Monetary가 0 이하인 고객 수

41


In [15]:
rfm = rfm.dropna()                  # R, F가 빈 61명 제거 (dropna = 빈칸 있는 줄 지우기)
print(len(rfm))
rfm = rfm[rfm['Monetary'] > 0]      # 순구매액이 0 이하인 고객 제거
print(len(rfm))

5881
5840


In [16]:
rfm.to_csv('/content/drive/MyDrive/marketing-portfolio/data/rfm.csv')

RFM 분석 대상에서 제외: ① 기간 내 취소 기록만 있는 고객(R, F 계산 불가) ② 구매 후 취소로 순구매액이 0 이하인 고객. 취소 이력이 있어도 순구매액이 플러스인 고객은 포함. =결국 머니터리가 음수인 고객을 제외하고 분석